# Decision Tree Classifier
# 16 June 2025

In [ ]:
import numpy as np
from collections import Counter

In [ ]:
# Class definition of a decision node (leaf node)
class DecisionNode:
    def __init__(self, feature_idx=None, threshold=None, left=None, right=None, value=None):
        self.feature_idx = feature_idx  # Index of feature to split on
        self.threshold = threshold     # Threshold value for the split
        self.left = left               # Left subtree (<= threshold)
        self.right = right             # Right subtree (> threshold)
        self.value = value             # Value if leaf node (for classification)


In [ ]:
# Calculate Gini impurity for a set of labels
def gini_impurity(y):
    counts = Counter(y)
    impurity = 1
    for lbl in counts:
        prob = counts[lbl] / len(y)
        impurity -= prob ** 2
    return impurity

In [ ]:
# Calculate information gain from a split
def information_gain(parent, left_child, right_child):
    n = len(parent)
    p_left = len(left_child) / n
    p_right = len(right_child) / n

    infoGain = gini_impurity(parent) - (p_left * gini_impurity(left_child) + \
                                        p_right * gini_impurity(right_child))
    return infoGain


In [ ]:
# Find the best split for a node
def find_best_split(X, y):
    best_gain = 0
    best_feature_idx, best_threshold = None, None

    n_features = X.shape[1]

    for feature_idx in range(n_features):
        thresholds = np.unique(X[:, feature_idx])
        for threshold in thresholds:
            left_indices = X[:, feature_idx] <= threshold
            right_indices = X[:, feature_idx] > threshold

            if len(y[left_indices]) == 0 or len(y[right_indices]) == 0:
                continue

            gain = information_gain(y, y[left_indices], y[right_indices])

            if gain > best_gain:
                best_gain = gain
                best_feature_idx = feature_idx
                best_threshold = threshold

    return best_feature_idx, best_threshold, best_gain

In [ ]:
# Recursively build the decision tree (tree induction function)
def build_tree(X, y, max_depth=None, min_samples_split=2, depth=0):
    n_samples, n_features = X.shape

    # Stopping conditions
    if (max_depth is not None and depth >= max_depth) or n_samples < min_samples_split or len(np.unique(y)) == 1:
        leaf_value = Counter(y).most_common(1)[0][0]
        return DecisionNode(value=leaf_value)

    # Find best split
    feature_idx, threshold, gain = find_best_split(X, y)

    if gain == 0:  # No gain from splitting
        leaf_value = Counter(y).most_common(1)[0][0]
        return DecisionNode(value=leaf_value)

    # Split the data
    left_indices = X[:, feature_idx] <= threshold
    right_indices = X[:, feature_idx] > threshold

    # Recursively build left and right subtrees
    left = build_tree(X[left_indices], y[left_indices], max_depth, min_samples_split, depth+1)
    right = build_tree(X[right_indices], y[right_indices], max_depth, min_samples_split, depth+1)

    return DecisionNode(feature_idx, threshold, left, right)

In [ ]:
# Preprocess the data: Convert categorical and handle missing values
def preprocess_data(df):
    # Convert sex to numerical values (M=0, F=1, I=2)
    df['Sex'] = df['Sex'].map({'M': 0, 'F': 1, 'I': 2})

    # Separate features and target
    X = df.drop('Rings', axis=1).values
    y = df['Rings'].values

    return X, y

In [ ]:
# Predict a single sample by traversing the tree
def predict_sample(tree, sample):
    if tree.value is not None:
        return tree.value

    if sample[tree.feature_idx] <= tree.threshold:
        return predict_sample(tree.left, sample)
    else:
        return predict_sample(tree.right, sample)

In [ ]:
# Predict for multiple samples
def predict(tree, X):
    return np.array([predict_sample(tree, sample) for sample in X])


In [ ]:
from google.colab import files
import os

import pandas as pd

In [ ]:
# Upload the file
uploaded = files.upload()

Saving abalone.csv to abalone.csv


In [ ]:
# Load the Excel or CSV File
# Change this to your actual file name
input_file = "abalone.csv"

file_ext = os.path.splitext(input_file)[1].lower()
if file_ext == '.csv':
    myDF = pd.read_csv(input_file)
elif file_ext in ['.xls', '.xlsx']:
    myDF = pd.read_excel(input_file)
else:
    raise ValueError("Unsupported file format. Use .csv or .xlsx")

# Print the first 50 rows for exploration
print("First 50 rows:\n", myDF.head(50))

First 50 rows:
    Sex  Length  Diameter  Height  WholeWeight  ShuckedWeight  VisceraWeight  \
0    M   0.455     0.365   0.095       0.5140         0.2245         0.1010   
1    M   0.350     0.265   0.090       0.2255         0.0995         0.0485   
2    F   0.530     0.420   0.135       0.6770         0.2565         0.1415   
3    M   0.440     0.365   0.125       0.5160         0.2155         0.1140   
4    I   0.330     0.255   0.080       0.2050         0.0895         0.0395   
5    I   0.425     0.300   0.095       0.3515         0.1410         0.0775   
6    F   0.530     0.415   0.150       0.7775         0.2370         0.1415   
7    F   0.545     0.425   0.125       0.7680         0.2940         0.1495   
8    M   0.475     0.370   0.125       0.5095         0.2165         0.1125   
9    F   0.550     0.440   0.150       0.8945         0.3145         0.1510   
10   F   0.525     0.380   0.140       0.6065         0.1940         0.1475   
11   M   0.430     0.350   0.110    

In [ ]:
import pandas as pd

abaloneDF = myDF.copy()
# # Load your data (replace this with your actual data loading)
# data = """Sex	Length	Diameter	Height	WholeWeight	ShuckedWeight	VisceraWeight	ShellWeight	Rings
# M	0.455	0.365	0.095	0.514	0.2245	0.101	0.15	15
# M	0.35	0.265	0.09	0.2255	0.0995	0.0485	0.07	7
# F	0.53	0.42	0.135	0.677	0.2565	0.1415	0.21	9
# M	0.44	0.365	0.125	0.516	0.2155	0.114	0.155	10
# I	0.33	0.255	0.08	0.205	0.0895	0.0395	0.055	7"""

# Preprocess data
X, y = preprocess_data(abaloneDF)

# Build the decision tree
tree = build_tree(X, y, max_depth=3)

# Make predictions
predictions = predict(tree, X)
print("Predictions:", predictions)
print("Actual values:", y)

Predictions: [ 8  9  8 ... 10 10 11]
Actual values: [15  7  9 ...  9 10 12]
